In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
df = pd.read_csv('dataset/AmesHousing.csv')

In [ ]:
df.head(15)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

In [ ]:
df.drop(columns=['Order', 'PID'], inplace=True)

In [ ]:
df.head(5)

In [ ]:
df.Alley.value_counts()

In [ ]:
sns.histplot(df['SalePrice'])

In [ ]:
df['MS SubClass'].unique()

In [ ]:
df['MS SubClass'].value_counts()

In [ ]:
sns.barplot(df['MS SubClass'].value_counts())

In [ ]:
sns.scatterplot(df, x='Gr Liv Area', y='SalePrice')

In [ ]:
skkew = df['SalePrice'].skew()

In [ ]:
print(skkew)

In [ ]:
df[df['MS SubClass']==150]

In [ ]:
df.drop(index=[2730], inplace=True)

In [ ]:
df.info(show_counts=True)

## Missing-Value Strategy

Verified empirically against the data (companion numeric columns prove absence):

### Bucket 1 — Absence (NaN = feature does not exist) → fill `'None'` / `0`
| Column(s) | Evidence |
|---|---|
| Alley, Pool QC, Fence, Misc Feature, Fireplace Qu | Pool QC NaN ⇒ Pool Area always 0; Fireplace Qu NaN ⇒ Fireplaces always 0 |
| Bsmt Qual/Cond/Exposure/BsmtFin Type 1+2 (+ SF numerics) | Bsmt Qual NaN ⇒ Total Bsmt SF = 0 |
| Garage Type/Finish/Qual/Cond + Garage Yr Blt | Garage Type NaN ⇒ Garage Area = 0, Garage Cars = 0 |

**Rows 1341 & 1497**: basement *absent* (not "lost") → basement numeric columns filled with 0.

### Bucket 2 — Genuinely missing → impute
| Column | Count | Choice | Reason |
|---|---|---|---|
| Lot Frontage | 490 | median per Neighborhood, fallback overall median | correlates with neighborhood; GrnHill & Landmrk have **no** Lot Frontage data at all |
| Mas Vnr Area | 23 | `0` | those rows also have Type = NaN (no veneer) |
| Mas Vnr Type | 1775 | `'None'` | 1745/1752 type-NaN rows have Area = 0. **Exception: 7 rows have Area > 0 (285–344)** — area says veneer exists, type unknown. `'None'` is a *documented* inconsistency, kept deliberately |
| Electrical | 1 (row 1577) | mode (`SBrkr`) | single case |
| Garage Area / Cars | 1 (row 2236) | median | garage *exists* (Detchd), value genuinely lost |

### Notes
- **Leakage caveat**: the groupby-median is fit on the full dataset. Fine for EDA; for final models move imputation into a `Pipeline` fit on train data only.
- **MS SubClass** converted to string — its values are category codes, not quantities.


In [ ]:
df['MS SubClass'] = df['MS SubClass'].astype(str)

In [ ]:
absence_cols = [
    'Alley', 'Pool QC', 'Fence', 'Misc Feature', 'Fireplace Qu',
    'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2',
    'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond','Mas Vnr Type',
]
df[absence_cols] = df[absence_cols].fillna('None')


df['Garage Yr Blt'] = df['Garage Yr Blt'].fillna(0)
basement_num = ['BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF',
                'Bsmt Full Bath', 'Bsmt Half Bath']
df[basement_num] = df[basement_num].fillna(0)

In [ ]:
df['Lot Frontage'] = df.groupby('Neighborhood')['Lot Frontage'].transform(lambda s: s.fillna(s.median()))
df['Lot Frontage'] = df['Lot Frontage'].fillna(df['Lot Frontage'].median())

df['Mas Vnr Area'] = df['Mas Vnr Area'].fillna(0)
df['Mas Vnr Type'] = df['Mas Vnr Type'].fillna('None')
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])
df['Garage Area'] = df['Garage Area'].fillna(df['Garage Area'].median())
df['Garage Cars'] = df['Garage Cars'].fillna(df['Garage Cars'].median())

In [ ]:
remaining = df.isna().sum()
remaining = remaining[remaining > 0]
print('columns still containing NaN:')
print(remaining if len(remaining) else 'NONE - 0 NaNs remain')
print('shape:', df.shape)

## Modelling Phase

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
model_df = pd.read_csv('dataset/AmesHousing.csv')

In [ ]:
model_df.shape

In [ ]:
model_df.drop(columns=['Order', 'PID'], inplace=True)

In [ ]:
model_df[model_df['MS SubClass'] == 150]
model_df.drop(index=[2730], inplace=True)

In [ ]:
model_df['MS SubClass'] = model_df['MS SubClass'].astype(str)

In [ ]:
train_df, test_df = train_test_split(model_df ,test_size=0.2, random_state=42)

In [ ]:
train_df.columns

In [ ]:
target_cols = 'SalePrice'
input_cols = list(train_df.columns)
input_cols.remove(target_cols)

In [ ]:
X_train , y_train = train_df[input_cols], train_df[target_cols]

In [ ]:
X_test , y_test = test_df[input_cols], test_df[target_cols]

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
'SalePrice' in X_train.columns

In [ ]:
model_df.isna().sum().sum() > 0

In [ ]:
X_test.shape

In [ ]:
y_test.shape

In [ ]:
none_imputer = SimpleImputer(fill_value='None', strategy='constant')

In [ ]:
X_train

In [ ]:
X_train.info()

In [ ]:
categorical_cols=list(X_train.select_dtypes(str).columns )
for col in absence_cols:
    if col in categorical_cols:
        categorical_cols.remove(col)

In [ ]:
other_categorical_impute = SimpleImputer(strategy='most_frequent')

In [ ]:
X_train

In [ ]:
numeric_cols = X_train.select_dtypes(np.number).columns
numeric_cols

In [ ]:
zero_cols = ['BsmtFin SF 1',
       'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF','Bsmt Full Bath',
       'Bsmt Half Bath','Garage Yr Blt', 'Mas Vnr Area']
median_cols = list(word for word in numeric_cols if word not in zero_cols)
median_cols

In [ ]:
zero_imputer = SimpleImputer(strategy='constant', fill_value=0)

In [ ]:
median_imputer = SimpleImputer(strategy='median')

In [ ]:
X_train.info()

In [ ]:
# --- Coverage gate: every feature in exactly one branch ---
import itertools
branches = {'A absence': absence_cols, 'B categorical': categorical_cols,
            'C zero': zero_cols, 'D median': median_cols}
for name, cols in branches.items():
    print(f'{name}: {len(cols)} cols')
print('sum:', sum(len(v) for v in branches.values()), '(expect 79)')

overlap = set()
for a, b in itertools.combinations(branches, 2):
    overlap |= set(branches[a]) & set(branches[b])
print('overlaps:', sorted(overlap) if overlap else 'NONE')
union = set().union(*(set(v) for v in branches.values()))
uncovered = set(input_cols) - union
stray = union - set(input_cols)
print('uncovered:', sorted(uncovered) if uncovered else 'NONE')
print('not-a-feature:', sorted(stray) if stray else 'NONE')

b_nan = X_train[categorical_cols].isna().sum()
print('raw NaNs remaining in branch B:')
print(b_nan[b_nan > 0] if (b_nan > 0).any() else '  NONE')

In [ ]:
# --- ColumnTransformer: 4 branches, each = impute -> process ---
none_pipe = Pipeline([
    ('impute', none_imputer),
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore')),
])
other_pipe = Pipeline([
    ('impute', other_categorical_impute),
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore')),
])
zero_pipe = Pipeline([
    ('impute', zero_imputer),
    ('scale', StandardScaler()),
])
median_pipe = Pipeline([
    ('impute', median_imputer),
    ('scale', StandardScaler()),
])

transformer = ColumnTransformer([
    ('none', none_pipe, absence_cols),
    ('other', other_pipe, categorical_cols),
    ('zero', zero_pipe, zero_cols),
    ('median', median_pipe, median_cols),
], remainder='drop')

In [ ]:
# fit on TRAIN only; test only ever gets transform (never fit)
X_train_t = transformer.fit_transform(X_train)
X_test_t = transformer.transform(X_test)
print('transformed shapes:', X_train_t.shape, X_test_t.shape)

In [ ]:
print('same feature count:', X_train_t.shape[1] == X_test_t.shape[1])
print('NaNs in train:', int(np.isnan(X_train_t).sum()), '| in test:', int(np.isnan(X_test_t).sum()))
print('79 raw features ->', X_train_t.shape[1], 'after one-hot + scaling')

In [ ]:
np.log1p(y_train)

In [ ]:
y_train.skew()

In [ ]:
y_train_raw, y_test_raw = y_train.copy(), y_test.copy()   # keep raw dollars for scoring
y_train = np.log1p(y_train)
y_test = np.log1p(y_test)

## Training

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso , Ridge, LassoCV, RidgeCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error,  mean_absolute_error, r2_score, mean_absolute_percentage_error

In [ ]:
def metricsShown(preds, y_test):
    print("MSE:", mean_squared_error(y_test, preds))
    print("MAE:", mean_absolute_error(y_test, preds))
    print("R2:", r2_score(y_test, preds))
    print("RMSE:", root_mean_squared_error(y_test, preds) )

In [ ]:
# Dont flag this
def R2Comp(preds_train, y_train, preds_test, y_test):
    print("Comparing R2 Score for checking learning")
    print("Train R2:", r2_score(y_train, preds_train), "Test R2:", r2_score(y_test, preds_test))

In [ ]:
def Modelling(model):
    model.fit(X_train_t, y_train)
    preds_train = np.exp(model.predict(X_train_t))
    preds_test  = np.exp(model.predict(X_test_t))
    print('train R2:', r2_score(y_train_raw, preds_train),
          '| test R2:', r2_score(y_test_raw, preds_test))
    metricsShown(preds_test, y_test_raw)
    return preds_test


### Linear Regression

In [ ]:
LinearReg = LinearRegression()

In [ ]:
Modelling(LinearReg)

### Ridge Regression

In [ ]:
RidgeReg = Ridge(random_state=42)

In [ ]:
Modelling(RidgeReg)

### Lasso Regression

In [ ]:
LassoReg = Lasso(random_state=42)

In [ ]:
Modelling(LassoReg)

### Decision Tree

In [ ]:
DecisionTree = DecisionTreeRegressor(random_state=42)

In [ ]:
Modelling(DecisionTree)

### Random Forest

In [ ]:
RandomForest = RandomForestRegressor(random_state=42)

In [ ]:
Modelling(RandomForest)

### XGBoost

In [ ]:
XGBoost = XGBRegressor(random_state=42)

In [ ]:
Modelling(XGBoost)

### LassoCV

In [ ]:
LassoCVReg = LassoCV(alphas=np.logspace(-5, -1, 50), cv=5, random_state=42, max_iter=5000)

In [ ]:
Modelling(LassoCVReg)

In [ ]:
F_L_kept = (LassoCVReg.coef_ != 0).sum()
print(f'{F_L_kept} of 328 features survived Lasso')

In [ ]:
print(f'alpha picked by LassoCV: {LassoCVReg.alpha_:.6f}')

### RidgeCV

In [ ]:
RidgeAlReg= RidgeCV(alphas=[0.01, 0.1, 1, 10, 100],cv=5)

In [ ]:
Modelling(RidgeAlReg)

In [ ]:
F_R_kept = (RidgeAlReg.coef_ != 0).sum()
print(f'{F_R_kept} of 328 features survived Ridge')

In [ ]:
print(f'alpha picked by RidgeCV: {RidgeAlReg.alpha_:.6f}')

## Table

In [ ]:
def mape(y_true, y_pred):    # sklearn gives fraction → ×100 for %
    return mean_absolute_percentage_error(y_true, y_pred) * 100

def smape(y_true, y_pred):   # no sklearn built-in — hand-roll, symmetric in y/ŷ
    return 100 * np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred)))

def adj_r2(y_true, y_pred, p):  # p = number of features = 328
    n = len(y_true)
    r2 = r2_score(y_true, y_pred)
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

In [ ]:
results  = {}
preds_store = {}

def evaluate(name, model):
    model.fit(X_train_t, y_train)
    preds_train = np.exp(model.predict(X_train_t))
    preds_test  = np.exp(model.predict(X_test_t))
    preds_store[name] = preds_test                    # keep for residuals/SHAP later
    results[name] = {
        'train R2': r2_score(y_train_raw, preds_train),
        'test R2':  r2_score(y_test_raw, preds_test),
        'MSE':  mean_squared_error(y_test_raw, preds_test),
        'RMSE': root_mean_squared_error(y_test_raw, preds_test),
        'MAE':  mean_absolute_error(y_test_raw, preds_test),
        'MAPE': mape(y_test_raw, preds_test),
        'SMAPE': smape(y_test_raw, preds_test),
        'Adjusted R2': adj_r2(y_train_raw, preds_train, X_train_t.shape[1]),
    }

# usage
evaluate('LinearRegression', LinearRegression())
evaluate('Ridge (a=1)',      Ridge(random_state=42))
evaluate('Lasso (a=1)',      Lasso(random_state=42))     # keep the collapsed one — it's your exhibit
evaluate('LassoCV',          LassoCVReg)
evaluate('RidgeCV',          RidgeAlReg)
evaluate('DecisionTree',     DecisionTree)
evaluate('RandomForest',     RandomForest)
evaluate('XGBoost',          XGBoost)

results_df = pd.DataFrame(results).T.sort_values('RMSE')
results_df

## Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_predict, KFold
kf = KFold(5, shuffle=True, random_state=42)

cv_results = {}
cv_preds_store = {}

def cv_evaluate(name, model):
    pipe = Pipeline([('prep', transformer), ('model', model)])
    log_pred = cross_val_predict(pipe, X_train, y_train, cv=kf)    # log space
    log_pred= np.clip(log_pred, y_train.min(), y_train.max())
    preds = np.expm1(log_pred)                                     # dollar space
    cv_preds_store[name] = preds
    cv_results[name] = {
        'CV log-R2':  r2_score(y_train, log_pred),        # ← model fit, fair
        'CV RMSE':    root_mean_squared_error(y_train_raw, preds),
        'CV R2 ($)':  r2_score(y_train_raw, preds),       # ← practical, outlier-fragile
        'max $pred':  preds.max(),                         # ← exposes extrapolation
    }

cv_evaluate('LinearRegression', LinearRegression())
cv_evaluate('Ridge (a=1)',      Ridge(random_state=42))
cv_evaluate('Lasso (a=1)',      Lasso(random_state=42))     # keep the collapsed one — it's your exhibit
cv_evaluate('LassoCV',          LassoCVReg)
cv_evaluate('RidgeCV',          RidgeAlReg)
cv_evaluate('DecisionTree',     DecisionTree)
cv_evaluate('RandomForest',     RandomForest)
cv_evaluate('XGBoost',          XGBoost)

cv_results_df = pd.DataFrame(cv_results).T.sort_values('CV RMSE')
cv_results_df

## Residuals

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
def residuals(name):
    resid = y_test_raw - preds_store[name]          # actual − predicted, in $

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].scatter(preds_store[name], resid, alpha=0.5); axes[0].axhline(0, color='r', lw=1)
    axes[0].set(xlabel='Predicted ($)', ylabel='Residual ($)', title='Residuals vs Predicted')
    axes[1].hist(resid, bins=40); axes[1].set(title='Residual distribution')
    axes[2].scatter(y_test_raw, preds_store[name], alpha=0.5)
    axes[2].plot([y_test_raw.min(), y_test_raw.max()]*2, 'r--')
    axes[2].set(xlabel='Actual ($)', ylabel='Predicted ($)', title='Predicted vs Actual')

In [ ]:
residuals('LinearRegression')

In [ ]:
residuals('Ridge (a=1)')

In [ ]:
residuals('Lasso (a=1)')

In [ ]:
residuals('LassoCV')

In [ ]:
residuals('RidgeCV')

In [ ]:
residuals('DecisionTree')

In [ ]:
residuals('RandomForest')

In [ ]:
residuals('XGBoost')

## SHAP

In [ ]:
import shap
feature_names = transformer.get_feature_names_out()
X_train_t_df = pd.DataFrame(X_train_t, columns=feature_names)
X_test_t_df  = pd.DataFrame(X_test_t,  columns=feature_names)   # explain the holdout


In [ ]:
# Tree model
explainer = shap.TreeExplainer(XGBoost)
sv = explainer.shap_values(X_test_t_df)
shap.summary_plot(sv, X_test_t_df, max_display=20)          # beeswarm: importance + direction
shap.summary_plot(sv, X_test_t_df, plot_type='bar')

In [ ]:
explainer = shap.TreeExplainer(RandomForest)
sv = explainer.shap_values(X_test_t_df)
shap.summary_plot(sv, X_test_t_df, max_display=20)          # beeswarm: importance + direction
shap.summary_plot(sv, X_test_t_df, plot_type='bar')

In [ ]:
explainer = shap.TreeExplainer(DecisionTree)
sv = explainer.shap_values(X_test_t_df)
shap.summary_plot(sv, X_test_t_df, max_display=20)          # beeswarm: importance + direction
shap.summary_plot(sv, X_test_t_df, plot_type='bar')

In [ ]:
explainer_l = shap.LinearExplainer(LinearReg, X_train_t_df)
sv_l = explainer_l.shap_values(X_test_t_df )
shap.summary_plot(sv_l, X_test_t_df, max_display=20)
shap.summary_plot(sv_l, X_test_t_df, plot_type='bar')

In [ ]:
explainer_l = shap.LinearExplainer(LassoReg, X_train_t_df)
sv_l = explainer_l.shap_values(X_test_t_df )
shap.summary_plot(sv_l, X_test_t_df, max_display=20)
shap.summary_plot(sv_l, X_test_t_df, plot_type='bar')

In [ ]:
explainer_l = shap.LinearExplainer(RidgeReg, X_train_t_df)
sv_l = explainer_l.shap_values(X_test_t_df )
shap.summary_plot(sv_l, X_test_t_df, max_display=20)
shap.summary_plot(sv_l, X_test_t_df, plot_type='bar')

In [ ]:
explainer_l = shap.LinearExplainer(LassoCVReg, X_train_t_df)
sv_l = explainer_l.shap_values(X_test_t_df )
shap.summary_plot(sv_l, X_test_t_df, max_display=20)
shap.summary_plot(sv_l, X_test_t_df, plot_type='bar')

In [ ]:
explainer_l = shap.LinearExplainer(RidgeAlReg, X_train_t_df)
sv_l = explainer_l.shap_values(X_test_t_df )
shap.summary_plot(sv_l, X_test_t_df, max_display=20)
shap.summary_plot(sv_l, X_test_t_df, plot_type='bar')

### SHAP vs Lasso: does SHAP agree with Lasso's feature selection?

LassoCV kept **116 of 328** features (L1 zeroes the rest). Now we ask whether XGBoost's SHAP
importance ranks those same features as important — and where they disagree. Distinct variable
names per model (`sv_xgb` / `sv_lasso`) so nothing goes stale.

In [ ]:
# distinct names per model — never overwrite a SHAP array you still need
sv_xgb   = shap.TreeExplainer(XGBoost).shap_values(X_test_t_df)
sv_lasso = shap.LinearExplainer(LassoCVReg, X_train_t_df).shap_values(X_test_t_df)

# model-agnostic importance: mean |SHAP| per feature, XGBoost
tree_imp = pd.Series(np.abs(sv_xgb).mean(0), index=feature_names).sort_values(ascending=False)

# which features Lasso actually kept (non-zero coefficient)
lasso_coefs = pd.Series(LassoCVReg.coef_, index=feature_names)
lasso_kept  = set(lasso_coefs[lasso_coefs != 0].index)
print(f'Lasso kept {len(lasso_kept)} of {len(feature_names)} features')

# --- 1) DUMMY-LEVEL overlap: XGBoost's top-20 one-hot columns vs Lasso survivors ---
top20   = tree_imp.head(20).index
kept    = [f for f in top20 if f in lasso_kept]
dropped = [f for f in top20 if f not in lasso_kept]
print(f'\n=== dummy level ===')
print(f'{len(kept)}/20 of XGBoost top-20 kept by Lasso')
print('  kept   :', kept)
print('  dropped:', dropped)

# --- 2) SOURCE-VARIABLE overlap: reunite the one-hot dummies with their parent column ---
def source_var(fname):
    body = fname.split('__', 1)[1]                     # drop the transformer branch prefix
    matches = [c for c in input_cols if body == c or body.startswith(c + '_')]
    return max(matches, key=len) if matches else body

tree_imp_var = tree_imp.groupby(tree_imp.index.map(source_var)).sum().sort_values(ascending=False)
kept_vars    = {source_var(f) for f in lasso_kept}
top20_var    = tree_imp_var.head(20).index
kept_v       = [v for v in top20_var if v in kept_vars]
dropped_v    = [v for v in top20_var if v not in kept_vars]
print(f'\n=== grouped by source variable (fairer: dummies reunited) ===')
print(f'{len(kept_v)}/20 of XGBoost top-20 variables kept by Lasso')
print('  kept   :', kept_v)
print('  dropped:', dropped_v)

In [ ]:
# visual: top-20 source variables, coloured by whether Lasso kept them
top20_df = pd.DataFrame({
    'importance': tree_imp_var.head(20),
    'Lasso kept': ['kept' if v in kept_vars else 'dropped' for v in top20_var],
}).iloc[::-1]

ax = top20_df['importance'].plot.barh(
    color=top20_df['Lasso kept'].map({'kept':'tab:blue','dropped':'tab:red'}), figsize=(9,7), legend=False)
ax.set_xlabel('mean |SHAP|  (XGBoost, log-price units)')
ax.set_title('XGBoost top-20 features — red = Lasso zeroed it')
plt.tight_layout()